In [1]:
import numpy as np
np.bool = np.bool_ 
import pandas as pd
import os
from tqdm import tqdm

import mxnet as mx
from mxnet import gluon
from mxnet import autograd
from mxnet import image

import sys
sys.path.append('resuneta/src')
sys.path.append('decode/FracTAL_ResUNet/models/semanticsegmentation')
sys.path.append('decode/FracTAL_ResUNet/nn/loss')
sys.path.append('autogeobound/MXNet-ResUNeta/')

from bound_dist import get_distance, get_boundary
from FracTAL_ResUNet import FracTAL_ResUNet_cmtsk
#from ftnmt_loss import ftnmt_loss
from datasets import *

from sklearn.metrics import matthews_corrcoef

import random

import rasterio
from rasterio.enums import ColorInterp
from rasterio.features import shapes
from scipy import ndimage

import matplotlib.pyplot as plt
%matplotlib inline

import higra as hg
import torch
import pickle

import itertools

import imageio.v3 as imageio
from skimage.color import label2rgb
import cv2
from skimage.segmentation import mark_boundaries
from skimage.segmentation import find_boundaries

from shapely.geometry import shape
import shapely
from shapely import Polygon, orient_polygons

import ast

import glob

import multiprocessing

In [2]:
import random
np.random.seed(0)
random.seed(0)

data = pd.read_csv('inference_data/inference_data.csv')
image_filenames = data[data["num_cultivated_pixels"] > 0]["image_filename"].tolist() # images that have at least one cultivated pixel
image_filenames = random.sample(image_filenames, 10)

# Part 1

In [3]:
graph = hg.get_8_adjacency_graph((512, 512))
topleft_indices = list(itertools.product([0, 512, 1024, 1536], [0, 512, 1024, 1536]))

def getResult(image_filename):
    
    image_number = image_filename[-9:-4]
    model_prediction = np.load(f"testing/pipeline-test/ml_preds/{image_number}.npy")

    trees_list = []
    altitudes_list = []
    for topleft_index in topleft_indices:
        x1, x2, y1, y2 = topleft_index[0], topleft_index[0]+512, topleft_index[1], topleft_index[1]+512
        bound = model_prediction[x1:x2, y1:y2] 
        edge_weights = hg.weight_graph(graph,bound,hg.WeightFunction.mean)
        tree, altitudes = hg.watershed_hierarchy_by_dynamics(graph, edge_weights)
        trees_list.append(tree)
        altitudes_list.append(altitudes) 
    
    with open(f"testing/pipeline-test/watershed_trees_and_altitudes/{image_number}.pkl", 'wb') as f:
        pickle.dump((trees_list, altitudes_list), f)

In [4]:
with multiprocessing.pool.ThreadPool(25) as pool:
    results = list(tqdm(pool.imap_unordered(getResult, image_filenames), total=len(image_filenames)))
    pool.close()
    pool.join()

100%|██████████| 10/10 [00:41<00:00,  4.17s/it]


# Part 2

In [3]:
T_values = [0.04, 0.005, 0.19, 0.02, 0.003, 0.09]
topleft_indices = list(itertools.product([0, 512, 1024, 1536], [0, 512, 1024, 1536]))

image_border = np.ones((2048, 2048))
image_border[1:-1, 1:-1] = 0

image_bottom_right_corner = np.zeros((2048, 2048))
image_bottom_right_corner[512:, 512:] = 1

def getResult(image_filename):
    
    image_number = image_filename[-9:-4]
    
    src = rasterio.open(image_filename)
    transform = src.transform

    CDL_filename = image_filename.replace("images", "CDL")
    CDL = imageio.imread(CDL_filename)

    with open(f"testing/pipeline-test/watershed_trees_and_altitudes/{image_number}.pkl", 'rb') as f:
        trees_list, altitudes_list = pickle.load(f)

    results_simplified = []
    Ts = []
    rankings = []
    for k, T in enumerate(T_values):
        pred_segmentation = np.zeros((2048, 2048))
        for j in range(len(topleft_indices)):
            topleft_index = topleft_indices[j]
            x1, x2, y1, y2 = topleft_index[0], topleft_index[0]+512, topleft_index[1], topleft_index[1]+512
            tree = trees_list[j]
            altitudes = altitudes_list[j]
            pred_segmentation_j = hg.labelisation_horizontal_cut_from_threshold(tree,altitudes,threshold=T).astype(float) 
            binary_field_extent_j = find_boundaries(pred_segmentation_j)==0
            pred_segmentation[x1:x2, y1:y2] = binary_field_extent_j
        ndimage.label(pred_segmentation, output=pred_segmentation)
        ndimage.grey_dilation(pred_segmentation, size=(3, 3), output=pred_segmentation)

        # remove small fields 
        unique_labels, counts = np.unique(pred_segmentation, return_counts=True)
        remove_mask = np.isin(pred_segmentation, unique_labels[counts < 300]) 
        pred_segmentation[remove_mask] = 0

        # remove fields that don't intersect with CDL cultivated land
        field_labels = np.unique(CDL*pred_segmentation) 
        remove_mask = ~np.isin(pred_segmentation, field_labels) # remove fields that don't intersect with CDL cultivated land
        pred_segmentation[remove_mask] = 0

        # remove fields at image border
        border_labels = np.unique(image_border*pred_segmentation)
        remove_mask = np.isin(pred_segmentation, border_labels)
        pred_segmentation[remove_mask] = 0

        # remove fields fully contained in top and left image margins (these will be computed in the overlapping neighboring tiles)
        bottom_right_corner_labels = np.unique(image_bottom_right_corner*pred_segmentation)
        remove_mask = ~np.isin(pred_segmentation, bottom_right_corner_labels)
        pred_segmentation[remove_mask] = 0
    
        results_T = [shape(s) for (s, v) in shapes(source=pred_segmentation, mask=pred_segmentation>0, transform=transform)]
        results_simplified_T = [polygon.simplify(tolerance=8, preserve_topology=True).buffer(-8) for polygon in results_T]
        results_simplified = results_simplified + results_simplified_T
        Ts = Ts + [T]*len(results_simplified_T)
        rankings = rankings + [k+1]*len(results_simplified_T)

    df = pd.DataFrame({'geometry':results_simplified, 'watershed_threshold':Ts, 'priority_ranking':rankings})
    df.to_csv(f'testing/pipeline-test/final_outputs/{image_number}.csv', index=False)

In [4]:
with multiprocessing.pool.ThreadPool(25) as pool:
    results = list(tqdm(pool.imap_unordered(getResult, image_filenames), total=len(image_filenames)))
    pool.close()
    pool.join()

  0%|          | 0/10 [00:00<?, ?it/s]<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing G

# Concatenate all field predictions into one file

In [5]:
# https://stackoverflow.com/questions/75756119/concatenate-10-000-csv-files-in-a-directory-using-python-pandas-too-slow
inputs = [f'testing/pipeline-test/final_outputs/{image_filename[-9:-4]}.csv' for image_filename in image_filenames]
with open("testing/pipeline-test/final_outputs.csv", "w") as output:
  first = True
  for file in inputs:
    with open(file, "r") as inputfile:
      for no, line in enumerate(inputfile, 1):
        if no == 1 and not first:
          continue
        first = False
        output.write(line)

In [6]:
for file in inputs:
    df = pd.read_csv(file)
    print(file)
    print(len(df))

testing/pipeline-test/final_outputs/25762.csv
48
testing/pipeline-test/final_outputs/50598.csv
2715
testing/pipeline-test/final_outputs/28182.csv
5649
testing/pipeline-test/final_outputs/02718.csv
3324
testing/pipeline-test/final_outputs/17372.csv
2055
testing/pipeline-test/final_outputs/34262.csv
1783
testing/pipeline-test/final_outputs/32600.csv
1829
testing/pipeline-test/final_outputs/27122.csv
3
testing/pipeline-test/final_outputs/52308.csv
893
testing/pipeline-test/final_outputs/20305.csv
4564
